# Dataset inspection
Run from the repository root or notebook folder. No package installation needed.


In [ ]:
from pathlib import Path
from collections import Counter
import re
import sys

root = Path.cwd()
if root.name == "notebook":
    root = root.parent
sys.path.insert(0, str(root / "src"))
from behavior_prediction.datasets import load_cruxeval, load_re2_export

crux = load_cruxeval(root / "data/raw/cruxeval/cruxeval.csv")
re2 = load_re2_export(root / "data/raw/re2/re2_bench_lite_output_prediction_prompts.jsonl")
print(f"CRUXEval: {len(crux)} | RE2-Bench-lite: {len(re2)}")
print("RE2 sources:", dict(Counter(r["source_dataset"] for r in re2)))
print("RE2 complexity groups:", dict(Counter(r["complexity_group"] for r in re2)))

## CRUXEval
Inputs and reference outputs are Python-expression text; they are not executed here.

In [ ]:
crux_index = 0
example = crux[crux_index]
print(example.id)
print("\nCODE\n" + example.code)
print("\nINPUT\n" + example.input)
print("\nREFERENCE OUTPUT\n" + repr(example.output))
print("\nAdditional context: none")

## RE2-Bench-lite
Shows task blocks when the worked-example boundary is explicit; otherwise shows the full prompt. Reference validity remains to be checked.

In [ ]:
re2_index = 1
example = re2[re2_index]
prompt = example["messages"][0]["content"]
task = prompt.rsplit("[/EXAMPLE]", 1)[-1]
print(example["id"], "|", example["source_dataset"], "|", example["complexity_group"])
if "[/EXAMPLE]" not in prompt:
    print("\nFULL PROMPT (worked examples and task; no explicit example boundary)\n" + prompt)
else:
    for tag in ("PYTHON", "INPUT", "STRUCTURE"):
        blocks = re.findall(r"\[" + tag + r"\](.*?)\[/" + tag + r"\]", task, re.S)
        print(f"\n{tag} ({len(blocks)} blocks)")
        for block in blocks:
            print(block.strip())
print("\nREFERENCE VALUE TYPE:", type(example["target_value"]).__name__)
print("REFERENCE VALUE:", repr(example["target_value"]))
print("SERIALIZED REFERENCE:", example["target_output"])
# print(prompt)  # Full original prompt, including instructions and worked example.

## RE2 checks
Flags overlap and are not a complete count of bad tasks. Nothing is excluded or repaired here.

In [ ]:
import json
from behavior_prediction.re2_inspection import inspect_re2

provenance = json.loads((root / "data/provenance.json").read_text())
re2_source = next(p for p in provenance if "/re2/" in p["path"])
checks = inspect_re2(re2, {r.id for r in crux}, root / re2_source["source_path"])
flags = ["cruxeval_source", "shared_cruxeval_id", "unexposed_address", "invalid_input_json", "double_encoded_stdin"]
for flag in flags:
    print(f"{flag}: {sum(r[flag] for r in checks)} / {len(checks)}")


In [ ]:
flag = "unexposed_address"  # Change to invalid_input_json or double_encoded_stdin
flagged = [r for r in checks if r[flag]]
for i, row in enumerate(flagged):
    print(i, row["id"])

case_index = 0
if flagged:
    case = flagged[case_index]
    original = next(r for r in re2 if r["id"] == case["id"])
    print("\nINPUT\n", case["input_text"])
    print("\nREFERENCE:", repr(original["target_value"]))
    print("PARSE ERROR:", case["parse_error"])
    if "upstream_stdin" in case:
        print("UPSTREAM STDIN:", repr(case["upstream_stdin"]))
    # print(original["messages"][0]["content"])  # Full code and context.

## Earlier manual audit (50 cases)
One-reviewer exploratory judgments from 6 September on the same export. Missing state needs case review; the checks above cannot detect it reliably. These counts must not be extrapolated to all RE2 records or added to the flags.

In [ ]:
audit = json.loads((root / "data/re2_audit_sample.json").read_text())
from behavior_prediction.datasets import sha256_file
assert sha256_file(root / re2_source["path"]) == audit["export_sha256"], "Audit refers to a different export"
print(dict(Counter(r["classification"] for r in audit["cases"])))
category = "underspecified"
cases = [r for r in audit["cases"] if r["classification"] == category]
for i, case in enumerate(cases):
    print(i, case["id"])
if cases:
    case = cases[0]  # Change index to inspect another judgment
    print("\nEVIDENCE:", case["note"])
    # print(next(r for r in re2 if r["id"] == case["id"])["messages"][0]["content"])

## Screened selection
Remove 44 overlaps and 10 reviewed problematic cases; repair 7 malformed inputs from upstream. References and object representations stay unchanged. This is not full reference validation; Avatar stdin encoding is unchanged.


In [ ]:
from behavior_prediction.re2_selection import prepare_re2, write_re2_selection

re2_selected, selection = prepare_re2(root / "data")
print({k: selection[k] for k in ("input_count", "retained_count", "excluded_count", "repaired_count")})
print(selection["exclusion_counts"])
for case in selection["excluded"]:
    if case["reason"] != "cruxeval_overlap":
        print(case["id"], "\n ", case["evidence"])


In [ ]:
for i, case in enumerate(selection["repairs"]):
    print(i, case["id"])

repair_index = 0  # Choose 0–6 to inspect each repair.
case = selection["repairs"][repair_index]
print(case["id"])
print("\nBEFORE\n" + case["before"])
print("\nAFTER\n" + case["after"])
print("\nREASON:", case["evidence"])


In [ ]:
write_re2_selection(re2_selected, selection, root / "data/processed/re2")
print("Saved screened JSONL and selection report to data/processed/re2")
